# M3C1 — Find a problem. Ask AI. Check the result.
Use your own notebook copy and the issued **telco_source.csv**. Follow this notebook from top to bottom: run → read → write → continue.
There are exactly three empty AI-code cells: **C04, C12, C14**. Other code cells are ready tools or small mentor demonstrations.
Code cells run with ▶. To answer a text note, double-click it, replace `...` with your words, then click outside. The mentor will demonstrate once.
Use your familiar AI helper. It writes code; **you run Colab**. If a task fails, stop at that task, share the exact error, expected result and actual result, and ask for one local repair or mentor help. Do not remove checks to obtain success.

## C00 — Ready tools

Press ▶ once. Keep the prepared source closed; read its result.

In [ ]:
#@title C00 — Ready tools
from pathlib import Path
import hashlib, json
import pandas as pd
import numpy as np

def show_other_checks():
    text = original["TotalCharges"].astype("string").str.strip()
    numeric = pd.to_numeric(text, errors="coerce")
    facts = {
        "Extra identical rows": int(original.duplicated().sum()),
        "Repeated customer numbers": int(original["customerID"].duplicated().sum()),
        "Negative monthly payments": int(original["MonthlyCharges"].lt(0).sum()),
        "Negative known total payments": int(numeric.lt(0).sum()),
        "Months outside the file's 0–72 range": int((original["tenure"].lt(0) | original["tenure"].gt(72)).sum()),
    }
    display(pd.DataFrame(list(facts.items()), columns=["What we checked", "Found"]))
    print("Words in OnlineSecurity:")
    display(original["OnlineSecurity"].value_counts(dropna=False).rename("customers"))
    print("Zero is a valid result. A surprising value needs a reason, not automatic deletion.")

def _strict_result_check(show=True):
    text = _original_snapshot["TotalCharges"].astype("string").str.strip()
    numeric = pd.to_numeric(text, errors="coerce")
    unknown = numeric.isna()
    same_rows = len(data) == len(_original_snapshot)
    same_ids = data.get("customerID", pd.Series(dtype=str)).tolist() == _original_snapshot["customerID"].tolist()
    same_churn = data.get("Churn", pd.Series(dtype=str)).tolist() == _original_snapshot["Churn"].tolist()
    same_original = original.equals(_original_snapshot)
    same_file = _source_path.is_file() and hashlib.sha256(_source_path.read_bytes()).hexdigest() == _source_digest
    allowed = set(_original_snapshot.columns) | {"TotalCharges_missing", "tenure_years"}
    same_other = set(data.columns) == allowed and same_rows
    if same_other:
        for column in _original_snapshot.columns:
            if column != "TotalCharges" and not data[column].equals(_original_snapshot[column]):
                same_other = False
    flag_ok = "TotalCharges_missing" in data and same_rows
    if flag_ok:
        flag_ok = data["TotalCharges_missing"].tolist() == unknown.astype(int).tolist()
    numeric_ok = "TotalCharges" in data and pd.api.types.is_numeric_dtype(data["TotalCharges"])
    values_ok = numeric_ok and same_rows
    if values_ok:
        actual = pd.to_numeric(data["TotalCharges"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
        wanted = numeric.fillna(replacement_value).to_numpy(dtype=float, na_value=np.nan)
        values_ok = bool(np.allclose(actual, wanted, equal_nan=False))
    years_ok = "tenure_years" in data and same_rows
    if years_ok:
        actual = pd.to_numeric(data["tenure_years"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
        wanted = _original_snapshot["tenure"].to_numpy(dtype=float) / 12
        years_ok = bool(np.allclose(actual, wanted, equal_nan=False))
    checks = {
        "The original file and original table are unchanged": same_file and same_original,
        "The same customers and Churn answers are still present": same_rows and same_ids and same_churn,
        "TotalCharges contains numbers and only the agreed replacements": numeric_ok and values_ok,
        "The mark still shows where the amount was unknown": bool(flag_ok),
        "Years equal months divided by 12": bool(years_ok),
        "Other columns and category meanings are unchanged": same_other,
    }
    if show:
        display(pd.DataFrame({"Check": list(checks), "Result": ["OK" if v else "Needs attention" for v in checks.values()]}))
        if all(checks.values()):
            print("The data checks passed. Now explain your own decision in C17.")
        else:
            print("Not finished yet. Read the row marked Needs attention.")
            print("Stop here. Share the exact failing check with the mentor and repair this local task before continuing.")
    return checks


def check_my_result(show=True):
    required = set(_original_snapshot.columns) | {"TotalCharges_missing", "tenure_years"}
    missing = sorted(required - set(data.columns))
    try:
        checks = _strict_result_check(show=False)
    except (KeyError, ValueError, TypeError, IndexError) as error:
        checks = {
            "The original file and original table are unchanged": False,
            "The same customers and Churn answers are still present": False,
            "TotalCharges contains numbers and only the agreed replacements": False,
            "The mark still shows where the amount was unknown": False,
            "Years equal months divided by 12": False,
            "Other columns and category meanings are unchanged": False,
        }
        if show:
            print("Unfinished: current table cannot pass full validation:", type(error).__name__)
    if show:
        display(pd.DataFrame({"Check": list(checks), "Result": ["OK" if v else "Needs attention" for v in checks.values()]}))
        if missing:
            print("Unfinished: missing required fields:", ", ".join(missing))
        if all(checks.values()):
            print("The data checks passed. Now explain your own decision in C17.")
        else:
            print("Unfinished. Stop here and share the failed checks and safe current evidence with the mentor.")
    return checks

def save_my_result():
    checks = check_my_result(show=False)
    if not all(checks.values()):
        print("Not saved: current data has failed validation. Stop here and ask the mentor to repair this local task.")
        if Path("telco_cleaned.csv").exists():
            print("An older CSV is still in Files. This run did NOT update it.")
            print("Do not submit that older file as the result of the current code. Repair, check and save again.")
        return False
    data.to_csv("telco_cleaned.csv", index=False)
    reopened = pd.read_csv("telco_cleaned.csv")
    if len(reopened) != len(data) or reopened["customerID"].tolist() != data["customerID"].tolist():
        raise ValueError("The saved file differs. Show this message to the mentor.")
    for column in ["TotalCharges", "TotalCharges_missing", "tenure_years"]:
        if not np.allclose(reopened[column], data[column]):
            raise ValueError("The saved numbers differ. Show the mentor: " + column)
    log = pd.DataFrame([
        {"action": "TotalCharges: text to number, mark, lesson replacement", "count": int(data["TotalCharges_missing"].sum()), "reason": "Unknown is not zero; original is kept"},
        {"action": "Remove rows", "count": 0, "reason": "No evidence justifies removal in this file"},
        {"action": "Merge categories", "count": 0, "reason": "Existing meanings are kept"},
        {"action": "Add tenure_years", "count": len(data), "reason": "Months / 12; original months kept"},
    ])
    log.to_csv("telco_cleaning_log.csv", index=False)
    Path("M3C1_cleaning_parameters.json").write_text(json.dumps({
        "replacement_value": replacement_value, "role": "classroom_explanation_only",
        "next_lesson": "Use the original source and refit preparation; do not train from a globally prepared export"
    }, indent=2), encoding="utf-8")
    print("Saved and opened again: telco_cleaned.csv")
    print("Keep the short log and parameter file with your work.")
    print("Also save your executed notebook and your own 1–2 page explanation.")
    return True

print("Ready. Next: C01. The prepared checking code is not today's learning task.")

### Additional help — ready tools

These tools check and save your work. Their internal code is not today’s learning task. Expand only if you need help.

## C01 — Open the table

Press ▶ and upload only **telco_source.csv**. Read the short table. One row is one customer. `original` preserves the source; `data` is the working copy. `tenure` is months with the company, `MonthlyCharges` is payment for one month, and `TotalCharges` is total payment recorded so far.

In [ ]:
#@title C01 — Open telco_source.csv
_source_path = Path("telco_source.csv")
if not _source_path.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Put telco_source.csv next to the notebook, then run C01 again.")
    print("Choose only telco_source.csv")
    files.upload()
if not _source_path.exists():
    raise FileNotFoundError("The file is not uploaded yet. Run C01 and choose telco_source.csv.")
_source_digest = hashlib.sha256(_source_path.read_bytes()).hexdigest()
if _source_digest != '16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91':
    raise ValueError("Please use the issued telco_source.csv unchanged. Ask the mentor for the file.")
original = pd.read_csv(_source_path)
_original_snapshot = original.copy(deep=True)
data = original.copy(deep=True)
replacement_value = 1422.1
print("Opened:", len(original), "customers and", len(original.columns), "columns.")
print("The lesson replacement is prepared. We will explain its use with a small example.")
display(original[["tenure", "MonthlyCharges", "TotalCharges"]].head())

## C02 — A small text-to-number example

Predict: which of `" 10 "`, `" "`, `"30"`, `"40"` has no known amount? Watch the mentor run this small example. A space is text. `NaN` means no known number, not zero.

In [ ]:
demo_text = pd.Series([" 10 ", " ", "30", "40"])
demo_numbers = pd.to_numeric(demo_text.str.strip(), errors="coerce")
display(pd.DataFrame({"written as text": demo_text, "read as a number": demo_numbers}))

## C03 — Mark first, then fill

The mentor chooses **20** only for this small example: halfway between 10 and 30. Predict the mark after filling. **1 = originally unknown; 0 = known.** The replacement is not an observed payment.

In [ ]:
demo_mark = demo_numbers.isna().astype(int)
demo_filled = demo_numbers.fillna(20)
display(pd.DataFrame({"before": demo_numbers, "was unknown": demo_mark, "after": demo_filled}))

## C04 — Ask AI to inspect

We saw that a space can hide an unknown amount. Predict: will an ordinary null count find it? Ask for inspection only. Do not send customer rows or IDs to the chat.

Try your own wording first. For support, copy this WHOLE box to your AI helper. No earlier output or CSV attachment is needed. Read the reply; paste only its Python code into the empty cell below, then press ▶.

```text
Help me inspect a customer table in Google Colab. I am a beginner.
The pandas DataFrame original is already loaded; data is its separate working copy.
One row is one customer. tenure means months with the company, MonthlyCharges
means payment for one month, and TotalCharges means total payment recorded so far.
Write one short Python cell using pandas as pd. Inspect original only; change no
table, file, row, customerID, Churn answer or other field. Print row and column
counts and the dtypes of these three fields. Separately count existing nulls,
space-only TotalCharges and nonempty text that cannot become a number.
Show up to five affected examples using only tenure, MonthlyCharges and
TotalCharges, without customerID. Explain what to look for. I run the code in
Colab; do not claim it has run. No CSV attachment or earlier output is needed.
```

In [ ]:
# Paste your AI inspection code here, then press ▶.

## C05 — Write your finding and uncertainty

Read the output immediately above. Replace each `...` yourself.
- The problem is in: ...
- My output shows this count: ...
- It matters because: ...
- My own observation: ...
- What I still do not know: ...

The table can show an unavailable amount; it does not confirm the true payment or its cause. Do not invent a reason. Explain why text in an amount field can prevent calculation.

## C06 — Read the category check

Press ▶. Read the words and their counts. Zero is a valid result when a check finds none.

In [ ]:
#@title C06 — Category counts
print("Words in OnlineSecurity:")
display(original["OnlineSecurity"].value_counts(dropna=False).rename("customers"))

## C07 — Decide about categories

`" Yes "` and `"Yes"` can mean the same answer with different spaces. `"No"` and `"No internet service"` mean different things. Use the output above and the data card. If words are already correct, say no change is needed.

My decision about categories and why: ...

## C08 — Read record and anomaly checks

Press ▶. An identical row and a repeated customer number are different checks. Read each count; zero is a valid finding.

In [ ]:
#@title C08 — Record and anomaly counts
_text = original["TotalCharges"].astype("string").str.strip()
_numeric = pd.to_numeric(_text, errors="coerce")
_facts = {
    "Extra identical rows": int(original.duplicated().sum()),
    "Repeated customer numbers": int(original["customerID"].duplicated().sum()),
    "Negative monthly payments": int(original["MonthlyCharges"].lt(0).sum()),
    "Negative known total payments": int(_numeric.lt(0).sum()),
    "Months outside the file's 0–72 range": int((original["tenure"].lt(0) | original["tenure"].gt(72)).sum()),
}
display(pd.DataFrame(list(_facts.items()), columns=["What we checked", "Found"]))
print("Zero is a valid result. A surprising value needs investigation, not automatic deletion.")

## C09 — Decide about records

A surprising amount needs investigation; it does not automatically justify deletion. Use the output immediately above.

Rows I removed (or zero): ...

Reason: ...

A suspicious case I kept or discussed: ...

## C10 — Approve one correction

Keep customers and other fields unchanged. The ready `replacement_value` was supplied by the mentor for this exercise from preserved training IDs; you did not choose it. It keeps the row usable in this classroom exercise, but the real payment remains unknown. Mark unknown amounts **before** filling.

My approved change: ...

I expect to see: ...

These things must stay unchanged: ...

## C11 — Make one complete safe request

Press ▶. Copy its WHOLE request output, including **Context**, to your AI helper. Nothing from earlier tables is needed; attach no CSV. Read the plan before using the code. It must convert → mark → fill only unknown amounts. If the proposal removes rows or changes other fields, ask for a smaller correction. Unexpected nonempty text means stop and ask the mentor.

In [ ]:
#@title C11 — Make the complete correction request
def correction_prompt(original, replacement_value):
    """Safe current aggregates only; reject unexpected text before requesting fill."""
    text = original['TotalCharges'].astype('string').str.strip()
    numeric = pd.to_numeric(text, errors='coerce')
    spaces = text.eq('').fillna(False)
    unexpected = numeric.isna() & text.notna() & ~spaces
    if unexpected.any():
        raise ValueError('Unexpected nonempty TotalCharges text found. Stop and ask the mentor; do not fill it.')
    dtypes = ', '.join(f'{c}: {original[c].dtype}' for c in ['tenure','MonthlyCharges','TotalCharges'])
    return f'''Help me make one approved correction in Google Colab. I am a beginner.
Context (current safe aggregates; no customer rows or IDs):
original is the preserved pandas DataFrame; data is its separate working copy.
Rows: {len(original)}. Columns: {len(original.columns)}.
Relevant dtypes: {dtypes}.
TotalCharges existing nulls: {int(original['TotalCharges'].isna().sum())}.
TotalCharges space-only values: {int(spaces.sum())}.
TotalCharges unexpected nonempty text: {int(unexpected.sum())}.
The actual ready variable replacement_value is {replacement_value!r}.
tenure means months with the company; MonthlyCharges is one month's payment;
TotalCharges is total payment recorded so far. Missing does not mean zero.
The mentor supplied replacement_value from the preserved training IDs. It is
a classroom replacement, not an observed payment. I did not choose this number.
First explain the plan in one or two simple sentences, then give one Python cell.
Use original TotalCharges, strip spaces and convert to numbers with pandas as pd.
If any unexpected nonempty text cannot become a number, stop with a clear error
and ask the mentor; do not impute it. BEFORE filling, add TotalCharges_missing
to data: 1 for an originally unknown amount, otherwise 0. Fill only unknown
TotalCharges with the ready replacement_value. Change only TotalCharges and
add that flag. Preserve original, source file, row count, row order, customerID,
Churn and every other field. Remove no rows and merge no categories.
Show counts before and after and up to five changed examples with before,
after and was_unknown, without customerID or any other customer fields.
I will run and inspect the code. Do not claim it has run. This whole request
contains the context: no CSV attachment or earlier table needs to be copied.'''

print(correction_prompt(original, replacement_value))

## C12 — Run the approved AI correction

Paste only the Python code returned for the complete request immediately above. Press ▶ yourself. Read its before/after display and counts.

In [ ]:
# Paste the ONE approved correction here, then press ▶.

## C13 — Describe the correction output

Use the output immediately above.

What changed: ...

What the mark still tells me: ...

What I still cannot claim about the real payment: ...

## C14 — Ask AI: months to years

The duration can be easier to read in years. Predict: 24 months / 12 = **2 years**; what should 12 months give? Ask for one new column and keep the original months.

Try your own wording first. For support, copy this WHOLE box to your AI helper. No earlier output or CSV attachment is needed. Read the reply; paste only its Python code into the empty cell below, then press ▶.

```text
Help me add one field to my pandas DataFrame data in Google Colab.
One row is one customer. tenure is the duration in months. The previous task
requested a TotalCharges correction and an originally-unknown flag; its result
has not yet passed full validation. Preserve CURRENT TotalCharges values and
TotalCharges_missing if present. Do not repeat conversion, marking or filling.
Write one short Python cell: data["tenure_years"] = data["tenure"] / 12.
Preserve original, the source file, every row, customerID, Churn, tenure,
TotalCharges, TotalCharges_missing and every other existing field. Do not
recalculate or fill anything else. Show the first five pairs of tenure and
tenure_years only. Explain how I can check 24 months / 12 = 2 years by hand.
I run the code; do not claim execution. No CSV attachment or earlier output is needed.
```

In [ ]:
# Paste the code for ONE new column here, then press ▶.

## C15 — Check one pair by hand

Choose one real pair from the output immediately above.

Months: ...

My calculation, months / 12: ...

Displayed years: ...

Do they agree? ...

### Additional help — one local repair

Share only the failed check, safe evidence, exact error if any, and expected result with the mentor or AI helper. Ask for the smallest fix preserving the agreed scope. Insert a code cell directly here. After the fix, include `lesson_checks = check_my_result()` and `current_facts = current_evidence(_original_snapshot, data, lesson_checks)` in that local cell. Run it and read its fresh output. This is support for the current task.

## C16 — Validate the full result

Press ▶. Read every check and the current evidence below it. If anything needs attention, stop here and share the failed check and this safe evidence with the mentor. Add a new code cell immediately HERE for one small repair, then run the repair and the check in that same local cell. Keep all checks; continue only when current validation passes.

In [ ]:
#@title C16 — Validate current work
def current_evidence(source, data, checks):
    """Show current safe evidence without asserting unfinished work is correct."""
    before = pd.to_numeric(source['TotalCharges'].astype('string').str.strip(), errors='coerce')
    after = pd.to_numeric(data['TotalCharges'].astype('string').str.strip(), errors='coerce') if 'TotalCharges' in data else None
    flag = pd.to_numeric(data.get('TotalCharges_missing', pd.Series(dtype=float)), errors='coerce')
    source_ids = set(source['customerID'])
    current_ids = set(data['customerID']) if 'customerID' in data else set()
    categories = [c for c in source if c not in ['customerID', 'TotalCharges']
                  and not pd.api.types.is_numeric_dtype(source[c])]
    category_changes = 'Unfinished: customer alignment is unavailable'
    if 'customerID' in data and data['customerID'].is_unique and current_ids <= source_ids and all(c in data for c in categories):
        left = source.set_index('customerID').loc[data['customerID'], categories].reset_index(drop=True)
        right = data[categories].reset_index(drop=True)
        category_changes = int((~(left.eq(right) | (left.isna() & right.isna()))).to_numpy().sum())
    facts = {
        'Original rows': len(source), 'Current rows': len(data),
        'Unknown TotalCharges before': int(before.isna().sum()),
        'Unknown TotalCharges now': int(after.isna().sum()) if after is not None else 'Unfinished: TotalCharges unavailable',
        'Current marks equal to 1': int(flag.eq(1).sum()) if 'TotalCharges_missing' in data else 'Unfinished: flag not added',
        'Original customers removed': len(source_ids - current_ids) if 'customerID' in data else 'Unfinished: customer field unavailable',
        'Category cells changed among retained customers': category_changes,
        'Current validation status': 'PASS' if all(checks.values()) else 'Unfinished: read the failed checks',
    }
    display(pd.DataFrame(list(facts.items()), columns=['Current evidence', 'Observed result']))
    if all(c in data for c in ['tenure', 'tenure_years']) and len(data):
        display(data[['tenure', 'tenure_years']].head(1))
    else:
        print('Months/years pair: Unfinished; a required field or row is unavailable.')
    return facts

lesson_checks = check_my_result()
current_facts = current_evidence(_original_snapshot, data, lesson_checks)

## C17 — Explain and discuss

Use the evidence immediately above; write here. Answer in your own words:
1. What problem did I find, and what evidence supports it?
2. What exact change did I ask AI to write, and why?
3. What changed, and what stayed the same?
4. How did I check one result myself?

My partner asked: ...

I explained: ...

AI use (not available / proposed code / I ran and checked it): ...

Use your notes for the existing **1–2 page justification**. Explain unknown amounts, the mentor-supplied classroom replacement and its limitation, category decisions and every removal (or zero). Your partner asks a question; they do not edit your assessed file.

## C18 — Save and reopen

Press ▶ after validation passes. This tool validates the current data again, saves and reopens it. Read the result. Download **telco_cleaned.csv** from Colab Files; keep the short log and parameter file. If saving fails, stop here and ask the mentor. An older CSV is not evidence that current work passed.

In [ ]:
#@title C18 — Save and reopen current checked work
saved = save_my_result()

## C19 — Submit your own work

Submit through the announced LMS route: **executed notebook + telco_cleaned.csv + your 1–2 page justification**. Save your notebook with outputs and notes. Follow the announced deadline. Keep **telco_source.csv** unchanged for the next lesson.

Final sentence: “I found ..., asked AI to ..., and checked it by ...”.

If access failed, record the exact last successful step. A mentor’s worked example is not your own successful run.